## Using harmony-py for subsetting/reformatting NASA Earthdata

### What is Harmony?

Data reduction services allow users to request only the data they want, in the format and projection they want

*specific transformation services are enabled for data sets on a priority basis - Harmony is a WIP*

Reference: https://harmony.earthdata.nasa.gov/

### What is harmony-py?

Harmony-Py provides a Python alternative to directly using Harmony’s RESTful API. It handles NASA Earthdata Login (EDL) authentication and optionally integrates with the CMR Python Wrapper by accepting collection results as a request parameter.

Reference: https://harmony-py.readthedocs.io/



#### Overview of today's tutorial

1. Import packages
2. Authenticate and initialize a Harmony client
3. Define and explore a dataset
4. Build a request
5. Request, Download and plot your data!

#### Requirements

Earthdata Login

python 3.9 or higher

installation of `harmony-py`

Other python packages: `datetime` , `rasterio` for plotting data and `json` for displaying information
    

### Background: installing harmony-py

I have already added harmony-py to my environment, but you should be sure you do as well or manually install the package:

`pip install -U harmony-py`

### 1. Import Packages

In [1]:
# Hiding my password as I type!
import getpass

# Help for reading a JSON output in readable form
import json
from IPython.display import display, JSON

# Working with the data
import datetime as dt
import rasterio
import rasterio.plot

from harmony import BBox, Client, Collection, Request, Environment, CapabilitiesRequest

### 2. Authentication and Harmony client initialization

In [2]:
EDL_username = input('EDL username?:')
EDL_password = getpass.getpass('EDL password?:')
harmony_client = Client(auth=(EDL_username, EDL_password))

EDL username?: fritzdi
EDL password?: ········


### 3. Dataset selection and services discovery

#### Define dataset

Define which data set you’d like to access using the collection concept-id (see [how-to](https://nsidc.github.io/NSIDC-Data-Cookbook/how-to-guides/get_concept-ids.html) for retrieving collection concept-ids). For this example, we’ll use SMAP L3 Radiometer Global Daily 36 km EASE-Grid Soil Moisture, Version 9 (SPL3SMP), C2938664585-NSIDC_CPRD.

In [3]:
collection = Collection(id='C2938664585-NSIDC_CPRD') 
dataset_short_name = 'SPL3SMP'

#### Discover subsetting capabilities

Harmony has different subsetting services depending on the data set of interest

In [4]:
capabilities_request = CapabilitiesRequest(short_name=dataset_short_name)

In [5]:
capabilities = harmony_client.submit(capabilities_request)

In [6]:
JSON(capabilities)

<IPython.core.display.JSON object>

### 4. Build a Request

Based on what is returned in the capabilities request JSON string for our data set, we can ask for particular subsetting!

We can do a spatial subset:

In [7]:
bbox = BBox(-93.4,30.3,-77.0,44.5)

We can do a temporal subset (this is passed as a dictionary with “start” and “stop” keys, and dates supplied as datetime objects):

In [8]:
temporal = {'start': dt.datetime(2023,12,12),'stop': dt.datetime(2023,12,13)}

We can do a format request to get a GeoTIFF:

In [9]:
format='image/tiff'

We can select particular variables from the data set for our GeoTIFF output:

(Here it is useful to know what you're looking for depending on the data set, but you can dig into the variable list from the JSON return)

In [10]:
variables = ['Soil_Moisture_Retrieval_Data_AM/soil_moisture']

Now add in all of your variables to the request:

In [11]:
request = Request(
    collection = collection,
    spatial = bbox,
    temporal = temporal,
    format = format,
    variables = variables
)

#### ALTERNATIVE: Build the Request all at once!

In [ ]:
# request = Request(
#     collection=Collection(id=dataset_short_name),
#     spatial=BBox(-93.4,30.3,-77.0,44.5),
#     temporal={
#         'start': dt.datetime(2023,12,12),
#         'stop': dt.datetime(2023,12,13)
#     },
#     variables = ['Soil_Moisture_Retrieval_Data_AM/soil_moisture'],
#     format='image/tiff'
# )

### 5. Request, download and plot your data!

#### Submit the Request

In [12]:
job_id = harmony_client.submit(request)
results = harmony_client.download_all(job_id, directory=".", overwrite=True)

In [13]:
harmony_client.wait_for_processing(job_id, show_progress=True)

 [ Processing: 100% ] |###################################################| [|]


The job_id can be used to check on the details of your job progress. If you are logged into Earthdata, you can see your jobs here: https://harmony.earthdata.nasa.gov/workflow-ui

In [14]:
print(job_id)

393975a9-e06a-4929-bfa0-9d106f7d2876


View the URLs for your staged data

In [ ]:
url = list(harmony_client.result_urls(job_id))
url

#### Download results to your local directory

In [ ]:
futures = harmony_client.download_all(job_id, directory=".")
filelist = [f.result() for f in futures]  # get filepaths

#### Plot your results!

In [ ]:
for f in filelist:
    rasterio.plot.show(rasterio.open(f),cmap='terrain')